# 20. PCA 기반 이상치 탐지 (PCA Anomaly Detection)

2차원 데이터를 `PCA(n_components=1)`로 압축한 뒤 `inverse_transform`으로 복원하여 원본과의 재구성 오차(Reconstruction Error)가 큰 샘플을 이상치로 탐지합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 정상 데이터(100개)와 이상치 데이터(5개) 생성
np.random.seed(42)
normal_data = np.random.normal(loc=0, scale=1, size=(100, 2))
anomalies = np.random.normal(loc=5, scale=5, size=(5, 2))
data = np.vstack([normal_data, anomalies])
df = pd.DataFrame(data, columns=['Feature 1', 'Feature 2'])

# 표준화 후 1차원 PCA 압축 및 2차원 복원
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df)

pca = PCA(n_components=1, random_state=42)
data_pca = pca.fit_transform(data_scaled)
data_reconstructed = pca.inverse_transform(data_pca)

In [ ]:
# 재구성 오차 계산 및 임계치(상위 95%) 기반 이상치 판별
reconstruction_error = np.sum((data_scaled - data_reconstructed) ** 2, axis=1)
threshold = np.percentile(reconstruction_error, 95)
is_anomaly = reconstruction_error > threshold

print(f'임계치(95%): {threshold:.4f}, 탐지된 이상치 수: {is_anomaly.sum()}개')

plt.figure(figsize=(7, 5))
plt.scatter(data_scaled[~is_anomaly, 0], data_scaled[~is_anomaly, 1], label='Normal', alpha=0.7)
plt.scatter(data_scaled[is_anomaly, 0], data_scaled[is_anomaly, 1], color='red', marker='x', s=90, label='Anomaly')
plt.title('PCA Reconstruction Error Anomaly Detection')
plt.xlabel('Scaled Feature 1')
plt.ylabel('Scaled Feature 2')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()